In [ ]:
%%time
!pip install -q rapidfuzz jellyfish unidecode

import os
from pathlib import Path

# Auto-detect where Kaggle extracted your dataset in /kaggle/input
DATA_ROOT = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'train_source1.tsv' in files:
        DATA_ROOT = Path(root).parent  # points to 'dataset/'
        break

if DATA_ROOT is None:
    raise FileNotFoundError("Could not find train_source1.tsv in /kaggle/input. Please verify your dataset is added in the right sidebar.")

TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR  = DATA_ROOT / 'test'
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"✅ Found dataset at: {DATA_ROOT}")
print(f"   Train files: {[f.name for f in TRAIN_DIR.glob('*.tsv')]}")
print(f"   Test files:  {[f.name for f in TEST_DIR.glob('*.tsv')]}")


In [3]:
import os, re, gc, json, warnings, time, random
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler
import jellyfish
from unidecode import unidecode

from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

import xgboost as xgb
import lightgbm as lgb
import torch

from tqdm.auto import tqdm
tqdm.pandas()

# Verify Kaggle GPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)

# HYPERPARAMETERS
TFIDF_TOP_K        = 25
TFIDF_MIN_SIM      = 0.08
RULE_MIN_TOKENS    = 2
VAL_SPLIT          = 0.15
NEG_RATIO          = 5
TRAIN_SAMPLE_SIZE  = 60000   # Kaggle 30GB RAM easily handles 60K entities!

print("✅ Configuration and GPU ready")


Device: cuda
GPU: Tesla T4
VRAM: 14.6 GB
✅ Configuration and GPU ready


In [4]:
%%time
print("Loading training data...")
train_s1 = pd.read_csv(TRAIN_DIR / 'train_source1.tsv', sep='\t')
train_s2 = pd.read_csv(TRAIN_DIR / 'train_source2.tsv', sep='\t')
train_s3 = pd.read_csv(TRAIN_DIR / 'train_source3.tsv', sep='\t')
train_gt = pd.read_csv(TRAIN_DIR / 'train_ground_truth.tsv', sep='\t')

print(f"\n{'Dataset':<20} {'Rows':>12}")
print("=" * 35)
for name, df in [('Train S1', train_s1), ('Train S2', train_s2), ('Train S3', train_s3), ('Train GT', train_gt)]:
    print(f"{name:<20} {len(df):>12,}")

n_singletons = train_gt['matched_entity_ids'].isna().sum()
n_matched = train_gt['matched_entity_ids'].notna().sum()
print(f"\nSingletons (no match): {n_singletons:,} ({n_singletons/len(train_gt)*100:.1f}%)")
print(f"Has matches:          {n_matched:,} ({n_matched/len(train_gt)*100:.1f}%)")


Loading training data...

Dataset                      Rows
Train S1                2,206,821
Train S2                5,034,616
Train S3                5,285,603
Train GT                2,206,821

Singletons (no match): 123,247 (5.6%)
Has matches:          2,083,574 (94.4%)
CPU times: user 30.1 s, sys: 2.99 s, total: 33.1 s
Wall time: 1min 21s


In [5]:
LEGAL_SUFFIXES = [
    'private limited', 'pvt ltd', 'pvt limited', 'limited liability partnership',
    'limited liability company', 'incorporated', 'corporation',
    'limited', 'inc', 'corp', 'llc', 'llp', 'lp', 'plc', 'ltd', 'pvt', 'co',
    'private', 'public',
    # Indic transliterated
    'praaivett limittedd', 'praivet limited', 'limittedd', 'praaivett', 'elelpii',
    'piraivet limitet', 'piraiveett limitett', 'limitet',
    # French
    'societe a responsabilite limitee', 'societe anonyme',
    'societe par actions simplifiee', 'societe civile immobiliere',
    'sarl', 'sas', 'sasu', 'eurl', 'sa', 'sci', 'gie', 'snc', 'scp',
    # Generic qualifiers
    'group', 'holding', 'holdings', 'enterprises', 'enterprise',
    'industries', 'industry', 'services', 'solutions', 'technologies',
    'technology', 'international', 'intl',
]

BUSINESS_PREFIXES = ['ms', 'the', 'ets', 'ste', 'cie', 'cabinet', 'groupe', 'dr', 'shri', 'messrs']

_SUFFIX_PATTERNS = [(re.compile(r'\b' + re.escape(s) + r'\b'), s) for s in sorted(LEGAL_SUFFIXES, key=len, reverse=True)]
_PREFIX_PATTERNS = [(re.compile(r'^\b' + re.escape(p) + r'\b\s*'), p) for p in sorted(BUSINESS_PREFIXES, key=len, reverse=True)]

ADDRESS_ABBR = {
    'rd': 'road', 'st': 'street', 'ave': 'avenue', 'blvd': 'boulevard',
    'dr': 'drive', 'ln': 'lane', 'ct': 'court', 'pl': 'place',
    'sq': 'square', 'hwy': 'highway', 'pkwy': 'parkway',
    'apt': 'apartment', 'ste': 'suite', 'fl': 'floor',
    'bldg': 'building', 'dept': 'department', 'mt': 'mount',
    'ft': 'fort', 'n': 'north', 's': 'south', 'e': 'east', 'w': 'west',
    'ne': 'northeast', 'nw': 'northwest', 'se': 'southeast', 'sw': 'southwest',
    'bd': 'boulevard', 'av': 'avenue', 'imp': 'impasse',
}

def normalize_text(text: str) -> str:
    if pd.isna(text) or not text:
        return ''
    text = unidecode(str(text)).lower().strip()
    text = text.replace('&', ' and ').replace('@', ' at ').replace('#', ' number ')
    text = re.sub(r'\bm/s\b|\bm\s*s\b', 'ms', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_name_components(name: str):
    if not name:
        return '', '', ''
    prefix = ''
    for pat, p in _PREFIX_PATTERNS:
        if pat.search(name):
            prefix = p
            name = pat.sub('', name).strip()
            break
    suffixes = []
    for pat, s in _SUFFIX_PATTERNS:
        if pat.search(name):
            suffixes.append(s)
            name = pat.sub('', name).strip()
    return re.sub(r'\s+', ' ', name).strip(), prefix, ' '.join(suffixes)

def expand_address_abbr(address: str) -> str:
    return ' '.join(ADDRESS_ABBR.get(t, t) for t in address.split())

def extract_numbers(text: str) -> str:
    return ' '.join(re.findall(r'\b\d+\b', text))

def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['business_name'] = df['business_name'].fillna('')
    df['business_address'] = df['business_address'].fillna('')
    df['country'] = df['country'].fillna('')
    
    df['clean_name'] = df['business_name'].apply(normalize_text)
    components = [extract_name_components(n) for n in df['clean_name']]
    df['clean_name_core'] = [c[0] for c in components]
    df['name_prefix']     = [c[1] for c in components]
    df['legal_suffix']    = [c[2] for c in components]
    df['clean_name_nosuffix'] = df['clean_name_core']
    
    df['clean_address'] = df['business_address'].apply(normalize_text).apply(expand_address_abbr)
    df['addr_numbers']  = df['clean_address'].apply(extract_numbers)
    df['combined_text'] = df['clean_name_core'] + ' | ' + df['clean_address']
    df['name_tokens']   = df['clean_name_core'].apply(lambda x: set(x.split()) if x else set())
    return df

print("✅ Preprocessing functions compiled")


✅ Preprocessing functions compiled


In [6]:
%%time
print("Preprocessing S1...")
train_s1 = preprocess_dataframe(train_s1)
train_s1['country'] = train_s1['country'].astype('category')
gc.collect()

print("Preprocessing S2...")
train_s2 = preprocess_dataframe(train_s2)
train_s2['country'] = train_s2['country'].astype('category')
gc.collect()

print("Preprocessing S3...")
train_s3 = preprocess_dataframe(train_s3)
train_s3['country'] = train_s3['country'].astype('category')
gc.collect()

print("Combining S2 and S3...")
train_s23 = pd.concat([train_s2, train_s3], ignore_index=True)
del train_s2, train_s3
gc.collect()

print("Setting lookups...")
train_s1.set_index('entity_id', drop=False, inplace=True)
train_s23.set_index('entity_id', drop=False, inplace=True)
train_s1_lookup = train_s1
train_s23_lookup = train_s23

print(f"✅ Preprocessed: S1={len(train_s1):,}, S2+S3={len(train_s23):,}")


Preprocessing S1...
Preprocessing S2...
Preprocessing S3...
Combining S2 and S3...
Setting lookups...
✅ Preprocessed: S1=2,206,821, S2+S3=10,320,219
CPU times: user 9min 48s, sys: 9.12 s, total: 9min 57s
Wall time: 9min 57s


In [7]:
def parse_ground_truth(gt_df):
    gt_dict = {}
    for _, row in gt_df.iterrows():
        s1_id = row['source1_entity_id']
        val = row['matched_entity_ids']
        if pd.isna(val) or not str(val).strip():
            gt_dict[s1_id] = set()
        else:
            gt_dict[s1_id] = set(m.strip() for m in str(val).split(',') if m.strip())
    return gt_dict

full_gt = parse_ground_truth(train_gt)
all_s1_ids = list(full_gt.keys())
random.seed(42)
random.shuffle(all_s1_ids)

split_idx = int(len(all_s1_ids) * (1 - VAL_SPLIT))
train_s1_ids = set(all_s1_ids[:split_idx])
val_s1_ids   = set(all_s1_ids[split_idx:])

train_gt_dict = {k: v for k, v in full_gt.items() if k in train_s1_ids}
val_gt_dict   = {k: v for k, v in full_gt.items() if k in val_s1_ids}

def compute_f05_macro(predictions: dict, ground_truth: dict) -> dict:
    f05_scores, precisions, recalls = [], [], []
    for s1_id, true in ground_truth.items():
        pred = predictions.get(s1_id, set())
        if isinstance(pred, list): pred = set(pred)
        if isinstance(true, list): true = set(true)
        
        if len(pred) == 0 and len(true) == 0:
            f05_scores.append(1.0); precisions.append(1.0); recalls.append(1.0); continue
        if len(pred) == 0:
            f05_scores.append(0.0); precisions.append(0.0); recalls.append(0.0); continue
        if len(true) == 0:
            f05_scores.append(0.0); precisions.append(0.0); recalls.append(1.0); continue
            
        tp = len(pred & true)
        fp = len(pred - true)
        fn = len(true - pred)
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f05 = (1.25 * p * r) / (0.25 * p + r) if (p + r) > 0 else 0.0
        f05_scores.append(f05); precisions.append(p); recalls.append(r)
        
    return {'f05': np.mean(f05_scores), 'precision': np.mean(precisions), 'recall': np.mean(recalls)}

print(f"✅ Split: Train S1 = {len(train_gt_dict):,}, Val S1 = {len(val_gt_dict):,}")


✅ Split: Train S1 = 1,875,797, Val S1 = 331,024


In [ ]:
%%time
def tfidf_blocking(s1_df, s23_df, top_k=25, min_sim=0.08, batch_size=200):
    candidates = {eid: set() for eid in s1_df['entity_id']}
    countries = s1_df['country'].dropna().unique()
    
    for c in countries:
        s1_c = s1_df[s1_df['country'] == c]
        s23_c = s23_df[s23_df['country'] == c]
        if len(s1_c) == 0 or len(s23_c) == 0: continue
            
        print(f"  [Country: {c}] S1: {len(s1_c):,} | S2+S3: {len(s23_c):,}")
        vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=120000, sublinear_tf=True, dtype=np.float32)
        s23_matrix = vectorizer.fit_transform(s23_c['combined_text'])
        s1_matrix  = vectorizer.transform(s1_c['combined_text'])
        
        s1_ids, s23_ids = s1_c['entity_id'].values, s23_c['entity_id'].values
        for start in tqdm(range(0, len(s1_ids), batch_size), desc=f"    TF-IDF Matching ({c})"):
            end = min(start + batch_size, len(s1_ids))
            sim_batch = (s1_matrix[start:end] @ s23_matrix.T).toarray()
            for j in range(sim_batch.shape[0]):
                row = sim_batch[j]
                above = np.where(row >= min_sim)[0]
                top_idx = above[np.argsort(row[above])[-top_k:]] if len(above) > top_k else above
                candidates[s1_ids[start + j]] = set(s23_ids[top_idx])
            del sim_batch
        del s23_matrix, s1_matrix, vectorizer
        gc.collect()
        
    return candidates

# Validation sample of 5,000 for quick recall evaluation
val_s1_sample = train_s1[train_s1['entity_id'].isin(val_s1_ids)].sample(min(5000, len(val_s1_ids)), random_state=42)
tfidf_cands_val = tfidf_blocking(val_s1_sample, train_s23, top_k=TFIDF_TOP_K, min_sim=TFIDF_MIN_SIM)


  [Country: US] S1: 2,955 | S2+S3: 6,186,873


In [1]:
%%time
def rule_based_blocking(s1_df, s23_df, min_shared_tokens=2):
    print("  Building token inverted index...")
    token_index = defaultdict(list)
    s23_countries = dict(zip(s23_df['entity_id'], s23_df['country']))
    
    for _, row in tqdm(s23_df.iterrows(), total=len(s23_df), desc="  Indexing S2+S3"):
        eid = row['entity_id']
        for token in row['name_tokens']:
            if len(token) > 2:
                token_index[token].append(eid)
                
    candidates = {}
    for _, row in tqdm(s1_df.iterrows(), total=len(s1_df), desc="  Rule matching"):
        s1_id, s1_c, tokens = row['entity_id'], row['country'], row['name_tokens']
        cand_counts = defaultdict(int)
        for token in tokens:
            if len(token) > 2:
                for cand_id in token_index.get(token, []):
                    cand_counts[cand_id] += 1
        candidates[s1_id] = {cid for cid, count in cand_counts.items() if count >= min_shared_tokens and s23_countries.get(cid) == s1_c}
    return candidates

rule_cands_val = rule_based_blocking(val_s1_sample, train_s23, min_shared_tokens=RULE_MIN_TOKENS)


NameError: name 'val_s1_sample' is not defined

In [ ]:
def merge_blocking_channels(*channels):
    all_ids = set().union(*[ch.keys() for ch in channels])
    merged = {s1_id: set().union(*[ch.get(s1_id, set()) for ch in channels]) for s1_id in all_ids}
    return merged

def evaluate_blocking_recall(candidates, ground_truth):
    total_true, found = 0, 0
    for s1_id, true_matches in ground_truth.items():
        if s1_id not in candidates: continue
        cands = candidates[s1_id]
        for m_id in true_matches:
            total_true += 1
            if m_id in cands: found += 1
    recall = found / total_true if total_true > 0 else 0
    avg_cands = sum(len(v) for v in candidates.values()) / max(1, len(candidates))
    return {'recall': recall, 'found': found, 'total_true': total_true, 'avg_cands': avg_cands}

merged_cands_val = merge_blocking_channels(tfidf_cands_val, rule_cands_val)

print("=" * 60)
for name, cands in [('TF-IDF', tfidf_cands_val), ('Rule-Based', rule_cands_val), ('Merged', merged_cands_val)]:
    res = evaluate_blocking_recall(cands, val_gt_dict)
    print(f"{name:<15} Recall: {res['recall']*100:.2f}% | Avg Candidates: {res['avg_cands']:.1f}")
print("=" * 60)


In [ ]:
def jaccard_tokens(a, b):
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0

def numbers_match(a, b):
    na, nb = set(re.findall(r'\b\d+\b', a)), set(re.findall(r'\b\d+\b', b))
    return 0.5 if not (na and nb) else (len(na & nb) / max(len(na), len(nb)))

def compute_pair_features(s1_row, cand_row):
    n1, n2   = s1_row['clean_name_core'], cand_row['clean_name_core']
    nf1, nf2 = s1_row['clean_name'], cand_row['clean_name']
    a1, a2   = s1_row['clean_address'], cand_row['clean_address']
    
    f = {}
    # Core Name features (Highest Weight)
    f['name_lev']        = fuzz.ratio(n1, n2) / 100.0
    f['name_jw']         = JaroWinkler.similarity(n1, n2)
    f['name_token_sort'] = fuzz.token_sort_ratio(n1, n2) / 100.0
    f['name_token_set']  = fuzz.token_set_ratio(n1, n2) / 100.0
    f['name_jaccard']    = jaccard_tokens(n1, n2)
    f['name_exact']      = float(n1 == n2)
    f['name_contain']    = float(n1 in n2 or n2 in n1) if n1 and n2 else 0.0
    f['name_full_jw']    = JaroWinkler.similarity(nf1, nf2)
    
    # Suffix & Prefix Compatibility (Secondary / Lower Weight)
    suf1, suf2 = s1_row['legal_suffix'], cand_row['legal_suffix']
    if not suf1 or not suf2: f['suffix_match'] = 0.5
    elif suf1 == suf2:       f['suffix_match'] = 1.0
    else:                    f['suffix_match'] = 0.0
        
    pre1, pre2 = s1_row['name_prefix'], cand_row['name_prefix']
    f['prefix_match'] = 1.0 if (not pre1 or not pre2 or pre1 == pre2) else 0.0
    
    # Address features
    if a1 and a2:
        f['addr_lev']        = fuzz.ratio(a1, a2) / 100.0
        f['addr_jw']         = JaroWinkler.similarity(a1, a2)
        f['addr_token_sort'] = fuzz.token_sort_ratio(a1, a2) / 100.0
        f['addr_jaccard']    = jaccard_tokens(a1, a2)
        f['addr_numbers']    = numbers_match(a1, a2)
    else:
        for k in ['addr_lev', 'addr_jw', 'addr_token_sort', 'addr_jaccard', 'addr_numbers']:
            f[k] = 0.0
            
    f['country_match'] = float(s1_row['country'] == cand_row['country'])
    f['has_addr_both'] = float(bool(a1) and bool(a2))
    return f

print(f"✅ Feature function defined (Features per pair: {len(compute_pair_features(train_s1.iloc[0], train_s23.iloc[0]))})")


In [ ]:
%%time
def build_training_pairs(s1_ids_list, gt_dict, candidates, s1_lookup, s23_lookup, neg_ratio=5):
    all_features, all_labels, all_pairs = [], [], []
    for s1_id in tqdm(s1_ids_list, desc="Building pairs"):
        if s1_id not in s1_lookup.index: continue
        s1_row = s1_lookup.loc[s1_id]
        true_matches = gt_dict.get(s1_id, set())
        cand_set = candidates.get(s1_id, set())
        
        # Positives
        for m_id in true_matches:
            if m_id in s23_lookup.index:
                all_features.append(compute_pair_features(s1_row, s23_lookup.loc[m_id]))
                all_labels.append(1)
                all_pairs.append((s1_id, m_id))
                
        # Hard Negatives
        hard_negs = [c for c in cand_set if c not in true_matches and c in s23_lookup.index]
        n_neg = min(len(hard_negs), max(neg_ratio * len(true_matches), neg_ratio))
        if n_neg > 0:
            for neg_id in random.sample(hard_negs, n_neg):
                all_features.append(compute_pair_features(s1_row, s23_lookup.loc[neg_id]))
                all_labels.append(0)
                all_pairs.append((s1_id, neg_id))
                
    X = pd.DataFrame(all_features).fillna(0)
    y = np.array(all_labels)
    return X, y, all_pairs

# 1. Build training blocking on a sample of 30,000 entities
train_sample_ids = list(train_s1_ids)[:30000]
train_sample_df  = train_s1.loc[train_sample_ids]

tfidf_cands_train = tfidf_blocking(train_sample_df, train_s23, top_k=TFIDF_TOP_K, min_sim=TFIDF_MIN_SIM)
rule_cands_train  = rule_based_blocking(train_sample_df, train_s23, min_shared_tokens=RULE_MIN_TOKENS)
merged_cands_train = merge_blocking_channels(tfidf_cands_train, rule_cands_train)

# 2. Build pairwise feature matrices
print("\nBuilding Train pairs...")
X_train, y_train, pairs_train = build_training_pairs(train_sample_ids, train_gt_dict, merged_cands_train, train_s1_lookup, train_s23_lookup, neg_ratio=NEG_RATIO)

print("\nBuilding Val pairs...")
val_sample_ids = list(val_s1_sample['entity_id'])
X_val, y_val, pairs_val = build_training_pairs(val_sample_ids, val_gt_dict, merged_cands_val, train_s1_lookup, train_s23_lookup, neg_ratio=NEG_RATIO)

print(f"\n✅ Ready for training: X_train={X_train.shape}, X_val={X_val.shape}")


In [ ]:
%%time
# 1. Train XGBoost on GPU
scale_pos = (y_train == 0).sum() / max(1, (y_train == 1).sum())

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos,
    'tree_method': 'hist',
    'device': 'cuda' if DEVICE == 'cuda' else 'cpu',
    'seed': 42
}

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)

print("🚀 Training XGBoost on GPU...")
xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=1000, evals=[(dval, 'val')], early_stopping_rounds=40, verbose_eval=100)

# 2. Train LightGBM
print("\n🚀 Training LightGBM...")
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

lgb_params = {
    'objective': 'binary', 'metric': 'average_precision',
    'max_depth': 8, 'learning_rate': 0.05, 'scale_pos_weight': scale_pos,
    'seed': 42, 'verbose': -1
}
lgb_model = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_val], num_boost_round=1000, callbacks=[lgb.early_stopping(40)])

print("✅ Both models trained successfully!")


In [ ]:
# Predict validation probabilities
xgb_val_probs = xgb_model.predict(dval)
lgb_val_probs = lgb_model.predict(X_val)
ensemble_val_probs = 0.5 * xgb_val_probs + 0.5 * lgb_val_probs

# Score only the validation entities we predicted for (the 5K sample)
val_gt_sample = {k: val_gt_dict[k] for k in val_sample_ids}

def eval_thresh(probs, pairs, gt_dict, threshold):
    preds = defaultdict(set)
    for idx, (s1_id, cand_id) in enumerate(pairs):
        if probs[idx] >= threshold:
            preds[s1_id].add(cand_id)
    return compute_f05_macro(dict(preds), gt_dict)

best_f05, best_threshold = 0, 0.5
for t in np.arange(0.2, 0.9, 0.05):
    res = eval_thresh(ensemble_val_probs, pairs_val, val_gt_sample, t)
    if res['f05'] > best_f05:
        best_f05 = res['f05']
        best_threshold = t
    print(f"Threshold: {t:.2f} | F0.5: {res['f05']:.4f} | Prec: {res['precision']:.4f} | Rec: {res['recall']:.4f}")

print(f"\n🏆 Best Threshold for F0.5 = {best_threshold:.2f} (F0.5 = {best_f05:.4f})")


In [ ]:
%%time
print("Loading Test Data...")
test_s1 = pd.read_csv(TEST_DIR / 'test_source1.tsv', sep='\t')
test_s2 = pd.read_csv(TEST_DIR / 'test_source2.tsv', sep='\t')
test_s3 = pd.read_csv(TEST_DIR / 'test_source3.tsv', sep='\t')

print("Preprocessing Test Data...")
test_s1 = preprocess_dataframe(test_s1)
test_s2 = preprocess_dataframe(test_s2)
test_s3 = preprocess_dataframe(test_s3)
test_s23 = pd.concat([test_s2, test_s3], ignore_index=True)
del test_s2, test_s3
gc.collect()

test_s1.set_index('entity_id', drop=False, inplace=True)
test_s23.set_index('entity_id', drop=False, inplace=True)

# 1. Test Blocking
print("\nRunning Test Blocking (Country-aware)...")
test_tfidf = tfidf_blocking(test_s1, test_s23, top_k=TFIDF_TOP_K, min_sim=TFIDF_MIN_SIM)
test_rule  = rule_based_blocking(test_s1, test_s23, min_shared_tokens=RULE_MIN_TOKENS)
test_cands = merge_blocking_channels(test_tfidf, test_rule)

# 2. Test Scoring
print(f"\nScoring candidates for {len(test_s1):,} test entities...")
predictions = {}
cand_pairs_records = []

for s1_id in tqdm(test_s1['entity_id'], desc="Predicting matches"):
    cand_ids = list(test_cands.get(s1_id, set()))
    if not cand_ids:
        predictions[s1_id] = set()
        continue
        
    s1_row = test_s1.loc[s1_id]
    feats = [compute_pair_features(s1_row, test_s23.loc[cid]) for cid in cand_ids if cid in test_s23.index]
    valid_cands = [cid for cid in cand_ids if cid in test_s23.index]
    
    if not feats:
        predictions[s1_id] = set()
        continue
        
    for cid in valid_cands:
        cand_pairs_records.append({'source1_entity_id': s1_id, 'candidate_entity_id': cid})
        
    X_test_batch = pd.DataFrame(feats).fillna(0)
    p_xgb = xgb_model.predict(xgb.DMatrix(X_test_batch))
    p_lgb = lgb_model.predict(X_test_batch)
    p_ens = 0.5 * p_xgb + 0.5 * p_lgb
    
    predictions[s1_id] = {valid_cands[i] for i, prob in enumerate(p_ens) if prob >= best_threshold}

# 3. Save matching_results.tsv
matching_rows = []
for s1_id in test_s1['entity_id']:
    m = predictions.get(s1_id, set())
    matching_rows.append({'source1_entity_id': s1_id, 'matched_entity_ids': ','.join(sorted(m)) if m else ''})

df_matching = pd.DataFrame(matching_rows)
df_matching.to_csv(OUTPUT_DIR / 'matching_results.tsv', sep='\t', index=False)

# 4. Save candidate_pairs.tsv
df_cands = pd.DataFrame(cand_pairs_records)
df_cands.to_csv(OUTPUT_DIR / 'candidate_pairs.tsv', sep='\t', index=False)

# 5. Create final ZIP
!cd /kaggle/working/output && zip -r /kaggle/working/submission.zip matching_results.tsv candidate_pairs.tsv

print("\n🎉 SUCCESS! Your submission files are ready:")
print(f"   1. /kaggle/working/output/matching_results.tsv ({len(df_matching):,} rows)")
print(f"   2. /kaggle/working/output/candidate_pairs.tsv  ({len(df_cands):,} rows)")
print(f"   3. /kaggle/working/submission.zip (DOWNLOAD THIS!)")
